# T02 — Video Reader Check

Checks `anyup/video.py` (task T02, see `docs/plan/P1_experiments/T02_video_reader.md`) with the real backbones.

1. Frame-folder reader vs the Exp 2/3 loader (same files, same pixels)
2. Chunked = direct: one 16-frame chunk through `iter_backbone_chunks` equals `bb(frames)` exactly
3. Per-frame chunk invariance: DINOv2 / Swin per-frame, chunk 16 vs chunk 4 vs one call
4. Full clip with VideoMAE: chunk shapes, last-chunk padding, constant peak GPU memory, overlap 0 vs 4
5. Video file: DAVIS `bear` encoded to mp4; decord reader, clear error without decord

Run all cells on a GPU runtime, then paste the **final summary cell's output** back.

In [ ]:
# ── Cell 1 · Repo ───────────────────────────────────────────────────────────
import os, sys, subprocess
REPO_URL    = "https://github.com/mu-az88/anyup.git"
REPO_BRANCH = "experiments-task02"   # branch that contains anyup/video.py
REPO_DIR    = "/content/anyup"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    # existing clone may be on another branch — switch to REPO_BRANCH and update it
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", REPO_BRANCH, f"origin/{REPO_BRANCH}"], check=True)
if "anyup" in sys.modules:
    raise RuntimeError("anyup was already imported in this runtime — Runtime → Restart session, then run again")
subprocess.run(["pip", "install", "-e", REPO_DIR, "-q"], check=True)
subprocess.run(["pip", "install", "transformers", "timm", "-q"], check=True)
# decord: same package as requirements.txt; VideoFileSource needs it (no fallback)
r = subprocess.run(["pip", "install", "eva-decord", "-q"], capture_output=True, text=True)
print("eva-decord install:", "ok" if r.returncode == 0 else r.stderr[-500:])
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(subprocess.run(["git", "-C", REPO_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout)

In [ ]:
# ── Cell 2 · Imports + config ───────────────────────────────────────────────
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image

from anyup.backbones import BackboneLoader
from anyup.video import (open_video, iter_backbone_chunks, chunk_boundaries,
                         FrameFolderSource, VideoFileSource, TensorSource)

DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
DAVIS_URL  = "https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip"
DAVIS_ZIP  = "/content/DAVIS-2017-trainval-480p.zip"
DAVIS_ROOT = "/content/DAVIS"
CLIP       = "bear"   # 82 frames, 854×480 (DAVIS-2017 train)
CHUNK      = 16       # ↓ frames per backbone call (iter_backbone_chunks default); memory. Dependent: SMALL_CHUNK, OVERLAP < CHUNK
SMALL_CHUNK = 4       # ↓ second chunk size for the per-frame invariance check
OVERLAP    = 4        # overlap for the VideoMAE overlap check; must be even (tubelet 2) and < CHUNK
VIDEO_FPS  = 24       # DAVIS frame rate, used for the mp4 encode
VIDEO_CRF  = 18       # x264 quality for the mp4 encode (lower = less codec loss)

# Chunk-invariance tolerance (same weights, same input, different batch size → GPU rounding only)
REL_TOL       = 1e-3      # max|a − b| / max|a| must be below this (DINOv2 batch 16 vs 4 measured 4.4e-4, T02 Q25)
DIAG_BATCHES  = [1, 2, 4, 8, 16]   # ↓ chunk sizes for the DINOv2 batch-size diagnostic (Cell 6); smaller = more calls, slower
EXACT_MIN_COS = 0.99999   # every frame's cosine(a, b) must be above this
MEM_TOL = 0.10        # peak GPU memory per chunk may vary by at most 10% of the first chunk's

RESULTS = []   # (check name, passed: bool | None for informational, detail str)
def record(name, passed, detail):
    RESULTS.append((name, passed, detail))
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")

def compare(a, b):
    a, b = a.float().cpu(), b.float().cpu()
    max_abs = (a - b).abs().max().item()
    cos = F.cosine_similarity(a.flatten(1), b.flatten(1), dim=1).min().item()   # worst frame
    return max_abs, cos

def compare_rel(a, b):
    """→ (max_abs, max|a|, relative error max_abs / max|a|, worst-frame cosine)."""
    max_abs, cos = compare(a, b)
    scale = a.float().abs().max().item()
    return max_abs, scale, max_abs / scale, cos

def run_all(bb, src, **kw):
    """All chunks → concatenated features/guides on CPU + list of metas."""
    feats, guides, metas = [], [], []
    for out in iter_backbone_chunks(bb, src, **kw):
        feats.append(out.features.cpu()); guides.append(out.guide.cpu()); metas.append(out.meta)
    return torch.cat(feats), torch.cat(guides), metas

print("Device:", DEVICE, "| backbones:", BackboneLoader.available())

In [ ]:
# ── Cell 3 · DAVIS (always downloaded from ETH) ─────────────────────────────
if not os.path.exists(os.path.join(DAVIS_ROOT, "ImageSets")):
    if not os.path.exists(DAVIS_ZIP):
        subprocess.run(["wget", "-q", "-O", DAVIS_ZIP, DAVIS_URL], check=True)
    subprocess.run(["unzip", "-q", DAVIS_ZIP, "-d", "/content/"], check=True)

CLIP_DIR   = Path(DAVIS_ROOT) / "JPEGImages/480p" / CLIP
ALL_FRAMES = sorted(CLIP_DIR.glob("*.jpg"))   # Exp 2/3 ordering
N = len(ALL_FRAMES)
print(f"clip={CLIP} frames={N} size={Image.open(ALL_FRAMES[0]).size}")

## 1 · Frame-folder reader vs Exp 2/3 loader

In [ ]:
# ── Cell 4 · FrameFolderSource vs sorted(glob) + PIL + ToTensor ──────────────
src = open_video(CLIP_DIR)
ok_type  = isinstance(src, FrameFolderSource)
ok_files = [Path(p) for p in src.files] == ALL_FRAMES
old = torch.stack([T.ToTensor()(Image.open(p).convert("RGB")) for p in ALL_FRAMES])   # (N, 3, 480, 854)
new = src.get(range(N))                                                               # (N, 3, 480, 854)
mx = (old - new).abs().max().item()
record("folder reader == Exp 2/3 loader", ok_type and ok_files and mx == 0 and len(src) == N,
       f"type={type(src).__name__} same_files={ok_files} n={len(src)} native_size={src.native_size} max_abs={mx:.1e}")
FRAMES = new   # (N, 3, 480, 854) float [0, 1], reused below

## 2 · Chunked = direct (one 16-frame chunk)

In [ ]:
# ── Cell 5 · iter_backbone_chunks on 16 frames == bb(frames) ─────────────────
frames16 = FRAMES[:16]   # (16, 3, 480, 854)
CONFIGS = [("videomae", {}), ("swin_t", dict(temporal_mode="per_frame")),
           ("swin_t", dict(temporal_mode="clip")), ("dinov2", {})]
for name, kw in CONFIGS:
    bb = BackboneLoader.load(name, device=DEVICE, **kw)
    direct = bb(frames16)
    outs = list(iter_backbone_chunks(bb, TensorSource(frames16), chunk_size=None))
    o = outs[0]
    ok = (len(outs) == 1 and torch.equal(o.features, direct.features) and torch.equal(o.guide, direct.guide)
          and o.meta["frame_index"] == direct.meta["frame_index"] and o.meta["n_pad"] == 0)
    mx, _ = compare(o.features, direct.features)
    label = name + (f" {kw['temporal_mode']}" if kw else "")
    record(f"chunked == direct, {label}", ok,
           f"n_chunks={len(outs)} features={tuple(o.features.shape)} max_abs={mx:.1e} frame_index={o.meta['frame_index']}")
    del bb, direct, outs, o; torch.cuda.empty_cache()

## 3 · Per-frame chunk invariance (full clip)

In [ ]:
# ── Cell 6 · chunk 16 vs chunk 4 vs one call on all N frames ────────────────
for name, kw in [("dinov2", {}), ("swin_t", dict(temporal_mode="per_frame"))]:
    bb = BackboneLoader.load(name, device=DEVICE, **kw)
    src = open_video(CLIP_DIR)
    f16, g16, m16 = run_all(bb, src, chunk_size=CHUNK)
    f4,  g4,  m4  = run_all(bb, src, chunk_size=SMALL_CHUNK)
    whole = bb(FRAMES).features.cpu()   # one call, batch = N frames
    idx_ok = [f for m in m16 for f in m["frame_index"]] == list(range(N)) == [f for m in m4 for f in m["frame_index"]]
    mx_a, sc, rel_a, cs_a = compare_rel(f16, f4)
    mx_b, _,  rel_b, cs_b = compare_rel(f16, whole)
    ok = idx_ok and rel_a < REL_TOL and cs_a > EXACT_MIN_COS and rel_b < REL_TOL and cs_b > EXACT_MIN_COS
    record(f"chunk invariance, {name}", ok,
           f"features={tuple(f16.shape)} chunks={len(m16)}/{len(m4)} indices_ok={idx_ok} max|f|={sc:.3g} | "
           f"16v4 max_abs={mx_a:.2e} rel={rel_a:.2e} min_cos={cs_a:.7f} | "
           f"16vWhole max_abs={mx_b:.2e} rel={rel_b:.2e} min_cos={cs_b:.7f}")
    del bb, f16, f4, whole; torch.cuda.empty_cache()

# DINOv2 batch-size diagnostic (INFO): error vs one call on all N frames, per chunk size,
# and chunk 4 run twice (is the same batch size deterministic?)
bb = BackboneLoader.load("dinov2", device=DEVICE)
src = open_video(CLIP_DIR)
whole = bb(FRAMES).features.cpu()
rows = []
for bs in DIAG_BATCHES:
    f, _, _ = run_all(bb, src, chunk_size=bs)
    mx, sc, rel, cs = compare_rel(whole, f)
    rows.append(f"bs={bs}: rel={rel:.2e} min_cos={cs:.7f}")
f4a, _, _ = run_all(bb, src, chunk_size=4)
f4b, _, _ = run_all(bb, src, chunk_size=4)
mx, _, rel, _ = compare_rel(f4a, f4b)
record("dinov2 batch-size diagnostic (vs one call)", None, " | ".join(rows) + f" | chunk4 twice: max_abs={mx:.1e} rel={rel:.1e}")
del bb, whole, f, f4a, f4b; torch.cuda.empty_cache()

## 4 · Full clip with VideoMAE: padding, memory, overlap

In [ ]:
# ── Cell 7 · VideoMAE on all N frames ───────────────────────────────────────
bb = BackboneLoader.load("videomae", device=DEVICE)
src = open_video(CLIP_DIR)

# overlap = 0, measure peak GPU memory per chunk
peaks, feats0, metas0 = [], [], []
torch.cuda.reset_peak_memory_stats()
for out in iter_backbone_chunks(bb, src):
    peaks.append(torch.cuda.max_memory_allocated() / 2**20)   # MiB, covers this chunk's forward
    feats0.append(out.features.cpu()); metas0.append(out.meta)
    del out; torch.cuda.reset_peak_memory_stats()
f0 = torch.cat(feats0)
for m in metas0:
    print(f"   chunk {m['chunk_id']}: start={m['chunk_start']} n_valid={m['n_valid']} n_pad={m['n_pad']} "
          f"t_out={m['t_out']} frame_index[0..-1]={m['frame_index'][0]}..{m['frame_index'][-1]} is_last={m['is_last']}")
exp_starts = list(range(0, N, 16))
last_valid = N - exp_starts[-1]
ok = ([m["chunk_start"] for m in metas0] == exp_starts
      and [f for m in metas0 for f in m["frame_index"]] == list(range(0, N, 2))
      and metas0[-1]["n_pad"] == 16 - last_valid and metas0[-1]["t_out"] == (last_valid + 1) // 2
      and tuple(f0.shape) == (len(range(0, N, 2)), 768, 14, 14))
record("videomae full clip, overlap 0", ok,
       f"N={N} features={tuple(f0.shape)} chunks={len(metas0)} last n_valid={last_valid} n_pad={metas0[-1]['n_pad']} "
       f"boundaries={chunk_boundaries(metas0)}")
spread = (max(peaks) - min(peaks)) / peaks[0]
record("videomae peak GPU memory constant across chunks", spread <= MEM_TOL,
       f"peak MiB per chunk={[round(p) for p in peaks]} spread={spread:.1%} (tol {MEM_TOL:.0%})")

# explicit overlap=0 == default
f0b, _, _ = run_all(bb, src, overlap=0)
record("videomae overlap=0 == default", torch.equal(f0, f0b), f"max_abs={(f0 - f0b).abs().max().item():.1e}")

# overlap = OVERLAP (informational: features differ because the backbone sees different context)
fo, _, metas_o = run_all(bb, src, overlap=OVERLAP)
idx_ok = [f for m in metas_o for f in m["frame_index"]] == list(range(0, N, 2))
record(f"videomae overlap={OVERLAP} indices", idx_ok and fo.shape == f0.shape,
       f"features={tuple(fo.shape)} chunks={len(metas_o)} starts={[m['chunk_start'] for m in metas_o]} "
       f"n_ctx={[m['n_ctx'] for m in metas_o]} t_out={[m['t_out'] for m in metas_o]}")
cos = F.cosine_similarity(fo.flatten(1), f0.flatten(1), dim=1)   # per feature frame
record(f"videomae overlap={OVERLAP} vs 0, per-frame cosine", None,
       f"mean={cos.mean():.4f} min={cos.min():.4f} (first chunk identical: {torch.equal(fo[:8], f0[:8])})")
del bb; torch.cuda.empty_cache()

## 5 · Video file (DAVIS `bear` → mp4)

In [ ]:
# ── Cell 8 · Encode, decord reader, clear error without decord ──────────────
MP4 = f"/content/{CLIP}.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-framerate", str(VIDEO_FPS),
                "-i", str(CLIP_DIR / "%05d.jpg"), "-c:v", "libx264", "-crf", str(VIDEO_CRF),
                "-pix_fmt", "yuv420p", MP4], check=True)

# (a) decord reader
src_d = open_video(MP4)
print("backend:", src_d.backend, "| fps:", src_d.fps, "| n:", len(src_d), "| size:", src_d.native_size)
vid_d = src_d.get(range(len(src_d)))   # (n, 3, H, W)
record("video file: decord backend used", src_d.backend == "decord", f"backend={src_d.backend}")
ok_len = len(src_d) == N and src_d.native_size == tuple(FRAMES.shape[-2:])
record("video file: decord frame count + size", ok_len,
       f"n={len(src_d)} jpeg n={N} size={src_d.native_size} fps={src_d.fps}")
if ok_len:
    d = (vid_d - FRAMES).abs()
    record(f"video file: mp4 (crf {VIDEO_CRF}) vs JPEG pixels (codec loss)", None, f"max_abs={d.max():.4f} mean_abs={d.mean():.5f}")

# (b) without decord: clear ImportError (no fallback)
saved = sys.modules.get("decord")
sys.modules["decord"] = None           # makes `import decord` raise ImportError
try:
    VideoFileSource(MP4)
    err = None
except ImportError as e:
    err = str(e)
finally:
    if saved is None: sys.modules.pop("decord", None)
    else: sys.modules["decord"] = saved
record("video file: no decord → clear ImportError", err is not None and "eva-decord" in err, f"error={err!r}")

# (c) runner on the mp4
bb = BackboneLoader.load("dinov2", device=DEVICE)
fv, _, mv = run_all(bb, src_d)
record("video file: runner on mp4 (dinov2)", tuple(fv.shape)[0] == len(src_d) and mv[0]["source"] == MP4 and mv[0]["fps"] == src_d.fps,
       f"features={tuple(fv.shape)} chunks={len(mv)} fps={mv[0]['fps']} native_size={mv[0]['native_size']}")
del bb; torch.cuda.empty_cache()

## Summary — paste this output back

In [ ]:
# ── Cell 9 · Summary ────────────────────────────────────────────────────────
import transformers, timm
try:
    import decord; decord_v = decord.__version__
except Exception as e:
    decord_v = f"unavailable ({type(e).__name__})"
print(f"torch={torch.__version__} torchvision={__import__('torchvision').__version__} "
      f"transformers={transformers.__version__} timm={timm.__version__} decord={decord_v} "
      f"gpu={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"clip={CLIP} N={N}")
for name, passed, detail in RESULTS:
    tag = "INFO" if passed is None else ("PASS" if passed else "FAIL")
    print(f"[{tag}] {name}: {detail}")
n_fail = sum(p is False for _, p, _ in RESULTS)
print(f"\n{len(RESULTS)} checks, {n_fail} failed")